# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsf-rawnak/FlyRankAI-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row (after I build my feature table) = one content item (`content_hash_id`), scored using
only what happened by a fixed decision moment.** The raw source table is a different grain — one
row per `report_date x client_hash_id x content_hash_id` — and I aggregate it up to one row per
content item before I score anything. Both grains get verified below with real queries.

**Which tables:** `fact_content_daily_performance` (the daily time series — this is where my
five features come from) and `dim_clients` (to check per-client history depth before I trust any
date window). I'm not touching `fact_content_query_90d` or `dim_content` this week — keeping the
slice small on purpose, per the assignment.

**Time window:** I'm iterating on **`month=2026-03`** (a mid-panel month, not the sealed final
month `_sample` = June 2026). Decision moment = end of March (2026-03-31). Feature window =
all of March (everything on or before the decision moment — "knowable"). For the deliberate
leakage trap in Section 3, I pull `month=2026-04` too, on purpose, as the future window the trap
column comes from — that's the whole point of the exercise.

**What I'd predict or rank (label/proxy):** same shape I named in ML-03 — a decline label, but
built properly this time as `prior period -> future period`, not the current-window proxy the
starter CSV uses. This week: `is_declining_label = 1` when April impressions fall more than 20%
below March impressions for that content item. That's an observed future outcome, not a rule
applied to the same window as the features — which is exactly the upgrade I flagged owing in
ML-02/ML-03.

**One thing I deliberately exclude:** any FlyRank product-computed field (`health_score`,
`priority_score`, `action_type`, refresh flags) — none of these ship in the release, so there's
nothing to accidentally pull in, but I'm naming the exclusion anyway per the data contract rule:
observable signals only, never a product decision.


In [1]:
%pip -q install duckdb

import os, getpass
import duckdb

# Token order: env var -> Colab Secret -> prompt (last resort).
# Store this as a Colab Secret named HF_TOKEN (key panel, left sidebar) so it never
# gets pasted into a cell — this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

# Cheap sanity check before anything else — COUNT(*) touches Parquet metadata, not data.
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:12} {n:>14,} rows")

# Confirm actual column names before trusting anything below — schema check, not a
# counted verification fact, just a safety net in case a column name differs.
print()
print("fact_content_daily_performance columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df()["column_name"].tolist())


Paste your Hugging Face READ token (hf_...): ··········
dim_clients             104 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily       78,835,655 rows

fact_content_daily_performance columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `content_hash_id` | Context | Grouping/joining only — pseudonym, never a feature |
| `client_hash_id` | Context | Grouping/joining, and for client-holdout splits later — never a feature |
| `report_date` | Context | Defines which window a row falls into; not fed to a model directly |
| `gsc_impressions` (March, summed) | Feature | Observed search impressions, fully known by the March 31 decision moment |
| `gsc_clicks` (March, summed) | Feature | Same — observed, known by the decision moment |
| `gsc_avg_position` (March, averaged) | Feature | Observed search position, known by the decision moment |
| `ctr_mar` (derived: clicks/impressions) | Feature | Derived only from March columns above — still knowable at decision time |
| `active_days_mar` (days with impressions in March) | Feature | Derived from March rows only |
| `gsc_impressions` (April, summed) | Label input | Used ONLY to build `is_declining_label` — a future outcome, not a feature. This is exactly the column the Section 3 trap deliberately misuses. |
| `is_declining_label` | Label | The target — 1 if April impressions fell >20% vs March. Observed future outcome, not a current-window proxy. |
| `health_score`, `priority_score`, `action_type`, any refresh flag | Excluded | Not shipped in this release at all — FlyRank's product decisions are deliberately withheld so I discover signal from evidence, not from copying an existing rule |


## 3. Verify it with queries (grain, counts, availability)

**Fact 1 — the grain.** I claimed the source table is one row per
`report_date x client_hash_id x content_hash_id`. If that's true, grouping by those three columns
and asking for groups with more than one row should come back empty.


In [3]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows returned (should be 0 if the grain holds): {len(grain_check)}")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows returned (should be 0 if the grain holds): 0


,report_date,client_hash_id,content_hash_id,n


**Fact 2 — slice row count and date span.** March 2026 should be a real, bounded slice —
not the whole 17-month panel and not empty.


In [4]:
march_span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()

march_span


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date,n_content_items,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


**Fact 3 — availability, filtered with `IS TRUE`.** The dictionary warns this flag is
**three-valued** (`TRUE` / `FALSE` / `NULL`) — a plain `= FALSE` or `NOT ga4_data_available`
silently mishandles the NULL rows. `IS TRUE` is the only safe filter.


In [5]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_march_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_unavailable_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_null_flag_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()

availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_march_rows,ga4_available_rows,ga4_unavailable_rows,ga4_null_flag_rows
0,9841378,413966,6408671,3018741


### Five features, max — from March only

Every feature below is aggregated **only** from March rows — nothing after the March 31 decision
moment touches these. That's what "knowable at the decision moment" means in practice, not just
in words.


In [6]:
features_mar = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)                                   AS impressions_mar,
        SUM(gsc_clicks)                                        AS clicks_mar,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_mar,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS active_days_mar
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
""").df()

features_mar["ctr_mar"] = features_mar["clicks_mar"] / features_mar["impressions_mar"]

print(f"{len(features_mar):,} content items with >=100 March impressions")
features_mar.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,441 content items with >=100 March impressions


,content_hash_id,client_hash_id,impressions_mar,clicks_mar,avg_position_mar,active_days_mar,ctr_mar
0,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,899.0,1.0,5.908100,31,0.001112
1,content_65c50dfe9d87a585,client_62f4a7e64f5e0096,3108.0,0.0,6.969536,30,0.000000
2,content_275b6f7f733016d4,client_62f4a7e64f5e0096,810.0,1.0,4.866123,29,0.001235
3,content_4dc944b7d0b65ecc,client_62f4a7e64f5e0096,134.0,0.0,5.012831,26,0.000000
4,content_92c381fbd361212e,client_62f4a7e64f5e0096,536.0,1.0,4.442543,29,0.001866


**One line per feature — knowable at the decision moment because:**

1. `impressions_mar` — a sum of GSC impressions already logged for dates on or before March 31;
   nothing here happens after the decision moment.
2. `clicks_mar` — same table, same window, same reasoning as impressions.
3. `avg_position_mar` — mean GSC position over March only; a page's position in March was fully
   observed by March 31.
4. `active_days_mar` — a count of March calendar days with at least one impression; entirely
   inside the feature window by construction.
5. `ctr_mar` — a derived ratio of the two March columns above; derived-from-safe-columns stays
   safe, same rule as the data contract's field-classification table.


### The trap

Now I pull **April** — the month right after my decision moment — and deliberately build the
label from it: `is_declining_label = 1` if April impressions fell more than 20% below March.
That's a proper future-window label. Then I do the thing the assignment wants me to catch myself
doing: I add April's own impressions in as a **feature**, right alongside the five honest ones.


In [7]:
april = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_apr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-04-01' AND report_date < DATE '2026-05-01'
    GROUP BY 1
""").df()

model_df = features_mar.merge(april, on="content_hash_id", how="inner")
model_df["is_declining_label"] = (
    model_df["impressions_apr"] < 0.8 * model_df["impressions_mar"]
).astype(int)

print(f"{len(model_df):,} content items with both March features and an April outcome")
print(f"Base rate of decline: {model_df['is_declining_label'].mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,441 content items with both March features and an April outcome
Base rate of decline: 51.7%


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

honest_cols = ["impressions_mar", "clicks_mar", "avg_position_mar", "active_days_mar", "ctr_mar"]
X_honest = model_df[honest_cols].fillna(0)
y = model_df["is_declining_label"]

honest_model = LogisticRegression(max_iter=1000).fit(X_honest, y)
honest_auc = roc_auc_score(y, honest_model.predict_proba(X_honest)[:, 1])
print(f"Honest ROC-AUC (five March-only features): {honest_auc:.3f}")

# THE TRAP — add April's own impressions in as a "feature." This is the exact column
# the label was built from, so the model doesn't need to learn anything; it can just
# read the answer straight off this one column.
leaky_cols = honest_cols + ["impressions_apr"]
X_leak = model_df[leaky_cols].fillna(0)

leaky_model = LogisticRegression(max_iter=1000).fit(X_leak, y)
leaky_auc = roc_auc_score(y, leaky_model.predict_proba(X_leak)[:, 1])
print(f"Leaky ROC-AUC (April impressions included as a 'feature'): {leaky_auc:.3f}")
print(f"Jump from adding the leak: +{leaky_auc - honest_auc:.3f}")


Honest ROC-AUC (five March-only features): 0.646
Leaky ROC-AUC (April impressions included as a 'feature'): 1.000
Jump from adding the leak: +0.354


**Caught it, deleted it, kept the honest number.** `impressions_apr` is now dropped —
it never belonged in the feature set; it's what the label was built from. The number I actually
report and carry forward is the honest ROC-AUC from the five March-only features, not the
inflated one from the column that let the model see its own answer.


## 4. Data limits

**Named limitation: this slice cannot separate "genuinely declining" from "the client simply
started tracking later" or "AI-search click-outs are too sparse to read anything into."**
Three concrete versions of that:

1. **Unbalanced panel.** History depth differs wildly by client — some have 17 months, some far
   less. A March-vs-April comparison is fair within a content item, I checked this directly: all 55 clients present in March also have April data (0 missing) — so this specific risk doesn't bite in this slice. It's still a real risk in principle for other month pairs or smaller clients, just not one that shows up here.
2. **`ga4_data_available` can be NULL, not just TRUE/FALSE** (verified above) — rows with a NULL
   flag are neither "engagement happened" nor "engagement didn't happen," they're unmeasured.
   I'm not using GA4 columns yet this week, so this doesn't bite the five features above, but it
   will the moment I add engagement signals.
3. **One month of "future" isn't much of a future.** April is the very next month after March —
   a real decline label deserves more runway than 30 days to rule out a single bad week. This
   week's label is a proof-of-concept for the leakage lesson, not the label I'd ship in a
   capstone-grade model.


In [9]:
# Backs up limitation #1 with a real number: how many clients in my March slice
# actually have a full April to compare against?

client_coverage = con.sql(f"""
    WITH march_clients AS (
        SELECT DISTINCT client_hash_id
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    ),
    april_clients AS (
        SELECT DISTINCT client_hash_id
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '2026-04-01' AND report_date < DATE '2026-05-01'
    )
    SELECT
        (SELECT COUNT(*) FROM march_clients) AS n_clients_in_march,
        (SELECT COUNT(*) FROM april_clients) AS n_clients_in_april,
        (SELECT COUNT(*) FROM march_clients m
         WHERE NOT EXISTS (SELECT 1 FROM april_clients a WHERE a.client_hash_id = m.client_hash_id)
        ) AS march_clients_missing_april
""").df()

client_coverage


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_clients_in_march,n_clients_in_april,march_clients_missing_april
0,55,61,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.